# Task 5 – Model Evaluation & Stability

This notebook evaluates the best model from Task 3, reports standard regression metrics, performs stability analysis via noise injection and feature dropout, and visualises feature importance.

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.ml.evaluation import RegressionEvaluator
import time, os, json, pandas as pd

spark = SparkSession.builder.getOrCreate()
spark.sparkContext.setLogLevel('ERROR')

# Load data and the preprocessing pipeline (same as Task 2)
data_path = os.path.join(os.getcwd(), 'data', 'yellow_tripdata_2025-*.parquet')
raw_df = spark.read.parquet(data_path)

# Re‑apply the preprocessing pipeline from Task 2
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler, StandardScaler, Pipeline
cat_cols = ['vendor_id', 'store_and_fwd_flag']
indexers = [StringIndexer(inputCol=c, outputCol=c+'_idx', handleInvalid='keep') for c in cat_cols]
encoders = [OneHotEncoder(inputCol=c+'_idx', outputCol=c+'_vec') for c in cat_cols]
num_cols = ['passenger_count', 'trip_distance', 'pickup_longitude', 'pickup_latitude',
                    'dropoff_longitude', 'dropoff_latitude', 'trip_duration',
                    'pickup_hour', 'pickup_day', 'pickup_month']
assembler = VectorAssembler(inputCols=num_cols + [c+'_vec' for c in cat_cols], outputCol='features_assembled')
scaler = StandardScaler(inputCol='features_assembled', outputCol='features', withMean=True, withStd=True)
pipeline = Pipeline(stages=indexers + encoders + [assembler, scaler])
pipeline_model = pipeline.fit(raw_df)
prepared_df = pipeline_model.transform(raw_df).select('features', 'fare_amount')
train_df, test_df = prepared_df.randomSplit([0.8, 0.2], seed=42)

# Load the best model from Task 3.
# For reproducibility we re‑train the best algorithm (assume Gradient‑Boosted Trees).
from pyspark.ml.regression import GBTRegressor
best_model = GBTRegressor(featuresCol='features', labelCol='fare_amount',
                                   seed=42, maxIter=50, maxDepth=10).fit(train_df)

# ---------- Evaluation Metrics ----------
evaluator = RegressionEvaluator(labelCol='fare_amount', predictionCol='prediction')
predictions = best_model.transform(test_df)
metrics = {
    'RMSE': evaluator.evaluate(predictions, {"metricName": "rmse"}),
    'MAE': evaluator.evaluate(predictions, {"metricName": "mae"}),
    'MSE': evaluator.evaluate(predictions, {"metricName": "mse"}),
    'R2': evaluator.evaluate(predictions, {"metricName": "r2"})
}
metrics_df = pd.DataFrame([metrics])
display(metrics_df)  # <--- Screenshot this baseline metrics table

## Stability Analysis – Noise Injection
We add a small Gaussian noise (σ = 0.01) to each numeric feature and re‑evaluate.

In [ ]:
# Add noise to numeric columns (excluding the label)
noise_df = test_df
for col in num_cols:
    noise_df = noise_df.withColumn(col, F.col(col) + F.randn(seed=42) * 0.01)

pred_noise = best_model.transform(noise_df)
noise_metrics = {
    'RMSE': evaluator.evaluate(pred_noise, {"metricName": "rmse"}),
    'MAE': evaluator.evaluate(pred_noise, {"metricName": "mae"}),
    'MSE': evaluator.evaluate(pred_noise, {"metricName": "mse"}),
    'R2': evaluator.evaluate(pred_noise, {"metricName": "r2"})
}
noise_metrics_df = pd.DataFrame([noise_metrics])
display(noise_metrics_df)  # <--- Screenshot this noise‑injection metrics table

## Stability Analysis – Feature Dropout
We zero‑out the `pickup_hour` feature (one of the engineered temporal features) and re‑evaluate.

In [ ]:
# Dropout: set pickup_hour to 0 for all rows
dropout_df = test_df.withColumn('pickup_hour', F.lit(0))
# Re‑assemble the feature vector with the same pipeline (must re‑fit because the pipeline expects raw columns)
dropout_prepared = pipeline_model.transform(dropout_df).select('features', 'fare_amount')
pred_dropout = best_model.transform(dropout_prepared)
dropout_metrics = {
    'RMSE': evaluator.evaluate(pred_dropout, {"metricName": "rmse"}),
    'MAE': evaluator.evaluate(pred_dropout, {"metricName": "mae"}),
    'MSE': evaluator.evaluate(pred_dropout, {"metricName": "mse"}),
    'R2': evaluator.evaluate(pred_dropout, {"metricName": "r2"})
}
dropout_metrics_df = pd.DataFrame([dropout_metrics])
display(dropout_metrics_df)  # <--- Screenshot this feature‑dropout metrics table

## Comparison Table
The three tables above (baseline, noise, dropout) are combined for a clear view of stability.

In [ ]:
combined = pd.concat([
    metrics_df.assign(Scenario='Baseline'),
    noise_metrics_df.assign(Scenario='Noise Injection'),
    dropout_metrics_df.assign(Scenario='Feature Dropout')
], ignore_index=True)
combined = combined[['Scenario', 'RMSE', 'MAE', 'MSE', 'R2']]
display(combined)  # <--- Screenshot this comparison table

## Explainability – Feature Importance
We extract the feature importances from the GBT model and plot the top 15 features.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
# Get the feature names from the VectorAssembler stage (second‑last stage)
assembler_stage = pipeline_model.stages[-2]
assembler_input_cols = assembler_stage.getInputCols()
# Expand one‑hot encoded categorical columns to their individual category names
feature_names = []
for col in assembler_input_cols:
    if col.endswith('_vec'):
        # Find the corresponding StringIndexerModel to get the category labels
        idx = next(i for i, s in enumerate(pipeline_model.stages) if isinstance(s, StringIndexer) and s.getOutputCol() == col.replace('_vec', '_idx'))
        categories = pipeline_model.stages[idx].labels
        base = col.split('_')[0]  # e.g., 'vendor' from 'vendor_id_vec'
        feature_names.extend([f'{base}_{cat}' for cat in categories])
    else:
        feature_names.append(col)
# Retrieve importance vector from the best GBT model
importances = best_model.featureImportances.toArray()
# Sort and plot the top 15 features
top_n = 15
sorted_idx = np.argsort(importances)[::-1][:top_n]
plt.figure(figsize=(10, 6))
plt.barh(np.arange(top_n), importances[sorted_idx])
plt.yticks(np.arange(top_n), [feature_names[i] for i in sorted_idx])
plt.gca().invert_yaxis()
plt.xlabel('Feature Importance')
plt.title('Top 15 Feature Importances from GBT Model')
plt.tight_layout()
plt.show()  # <--- Screenshot this chart

If `matplotlib` is not available, replace the plotting section with a simple `print(importances)` and discuss the most important features.